# lora_check — consolidated results for the LoRA arm
Reads `scores/` only; generates nothing. The main LoRA result tables and diagnostics are reproduced here from the frozen score files, without re-running generation or the judge. The step-1000 rank-versus-rank TOST values reported in the notes (r4-r8 0.0005, r4-r16 0.0001, r8-r16 0.0029, widest 90% CI bound 0.093) are produced in section 10 below, together with the zero-reference interaction.

Covers: headline table, the steps effect (1000 vs 4000), the adapter against the prompting baseline, rank, TOST against the pre-registered ±0.15 bound, cluster-robust standard errors, stratification by reference count / provenance / reference resolution, the validation curves and the val-loss-versus-PF/SC disagreement, val-to-test transfer, and the zero-reference adaptation tax.

## 1. Load and integrity

In [1]:
from google.colab import drive
import os, json, math, re, itertools
from collections import defaultdict
import numpy as np
from scipy import stats
drive.mount("/content/drive")

DRIVE  = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
SCORES = f"{DRIVE}/scores"
RANKS  = (4, 8, 16)

def load_by_uid(path):
    d = {}
    for l in open(path, encoding="utf-8"):
        r = json.loads(l)
        if "error" in r: continue
        d[r["uid"]] = r
    return d

ov  = lambda r: math.sqrt(max(r["PF"], 0) * max(r["SC"], 0))
MET = [("PF", lambda r: r["PF"]), ("SC", lambda r: r["SC"]), ("Overall", ov)]

# test set, 488 rows
BASE  = load_by_uid(f"{SCORES}/test__baseline.jsonl")              # prompting baseline, references supplied
K4    = {r: load_by_uid(f"{SCORES}/test__lora_r{r}.jsonl")             for r in RANKS}   # adapter @ step 4000
K1    = {r: load_by_uid(f"{SCORES}/test__lora_r{r}_step1000.jsonl")    for r in RANKS}   # adapter @ step 1000
ZBASE = load_by_uid(f"{SCORES}/test__zero_ref.jsonl")              # base model, references removed
ZAD   = load_by_uid(f"{SCORES}/test__lora_r8_step1000_zero_ref.jsonl")  # adapter, references removed

# val slice, 98 rows
VBASE = load_by_uid(f"{SCORES}/val__baseline_val.jsonl")
VAL   = {r: {s: load_by_uid(f"{SCORES}/val__lora_{r}_step{s}.jsonl") for s in (1000, 2000, 3000, 4000)}
         for r in ("r4", "r8", "r16")}
VEARLY= {s: load_by_uid(f"{SCORES}/val__lora_r8_early_step{s}.jsonl") for s in (250, 500, 750, 1000)}

META  = {r["output_image"] + "|" + "|".join(sorted(r["input_images"])): r
         for r in [json.loads(l) for l in open(f"{DRIVE}/test.jsonl", encoding="utf-8")]}

I  = sorted(set(BASE) & set.intersection(*[set(d) for d in K1.values()]) & set.intersection(*[set(d) for d in K4.values()]))
IZ = sorted(set(ZBASE) & set(ZAD))
IV = sorted(set(VBASE) & set.intersection(*[set(d) for r in VAL.values() for d in r.values()]))

def guessed(path):
    n = 0
    for l in open(path, encoding="utf-8"):
        r = json.loads(l)
        if str(r.get("PF_reason", "")).startswith("guess_if_cannot_parse") \
           or str(r.get("SC_reason", "")).startswith("guess_if_cannot_parse"): n += 1
    return n

files = ["test__baseline"] + [f"test__lora_r{r}" for r in RANKS] + [f"test__lora_r{r}_step1000" for r in RANKS] \
        + ["test__zero_ref", "test__lora_r8_step1000_zero_ref"]
print(f"test rows paired across baseline and all six adapter runs : {len(I)}")
print(f"zero-reference rows paired                                 : {len(IZ)}")
print(f"val rows paired across all ranks and checkpoints           : {len(IV)}")
print(f"guessed judge scores across every file above               : {sum(guessed(f'{SCORES}/{f}.jsonl') for f in files)}")

Mounted at /content/drive
test rows paired across baseline and all six adapter runs : 488
zero-reference rows paired                                 : 488
val rows paired across all ranks and checkpoints           : 98
guessed judge scores across every file above               : 0


## 2. Headline — test set, 488 rows

In [2]:
def agg(d, idx=None):
    ks = idx or list(d)
    return tuple(np.mean([f(d[k]) for k in ks]) for _, f in MET)

print(f"{'run':26s} {'PF':>7s} {'SC':>7s} {'Overall':>8s}")
print(f"{'prompting baseline':26s} " + " ".join(f"{v:7.3f}" for v in agg(BASE, I)))
for r in RANKS:
    print(f"{f'LoRA r{r} @ step 1000':26s} " + " ".join(f"{v:7.3f}" for v in agg(K1[r], I)))
for r in RANKS:
    print(f"{f'LoRA r{r} @ step 4000':26s} " + " ".join(f"{v:7.3f}" for v in agg(K4[r], I)))
print()
print(f"{'base, no references':26s} " + " ".join(f"{v:7.3f}" for v in agg(ZBASE, IZ)))
print(f"{'LoRA r8 @1000, no refs':26s} " + " ".join(f"{v:7.3f}" for v in agg(ZAD, IZ)))

run                             PF      SC  Overall
prompting baseline           8.004   8.088   7.963
LoRA r4 @ step 1000          8.027   7.855   7.848
LoRA r8 @ step 1000          8.064   7.846   7.843
LoRA r16 @ step 1000         8.033   7.865   7.852
LoRA r4 @ step 4000          8.051   7.332   7.516
LoRA r8 @ step 4000          8.055   7.432   7.595
LoRA r16 @ step 4000         8.012   7.350   7.526

base, no references          7.238   5.035   5.727
LoRA r8 @1000, no refs       7.262   5.125   5.791


## 3. Paired contrasts — steps, baseline, rank
Every contrast is paired per row on content uid. `sigma_d` is the per-row spread of the delta; SE is `sigma_d / sqrt(n)`.

In [3]:
def paired(A, B, idx, f):
    d = np.array([f(A[k]) - f(B[k]) for k in idx], float)
    n = len(d); se = d.std(ddof=1) / math.sqrt(n)
    p = stats.ttest_rel([f(A[k]) for k in idx], [f(B[k]) for k in idx])[1]
    tc = stats.t.ppf(0.95, n - 1)
    return dict(n=n, mean=d.mean(), sd=d.std(ddof=1), se=se, p=p,
                lo=d.mean() - tc * se, hi=d.mean() + tc * se, d=d)

def show(lab, A, B, idx=None, mets=MET):
    idx = idx or I
    for name, f in mets:
        s = paired(A, B, idx, f)
        print(f"  {lab:30s} {name:8s} n={s['n']:4d}  delta {s['mean']:+7.3f}  sigma_d {s['sd']:5.3f}  "
              f"SE {s['se']:.3f}  p {s['p']:.3g}  90%CI [{s['lo']:+.3f}, {s['hi']:+.3f}]")

print("steps effect on test — step 1000 minus step 4000:")
for r in RANKS: show(f"r{r}  1000 - 4000", K1[r], K4[r])
print("\nadapter against the prompting baseline:")
for r in RANKS: show(f"r{r} @1000 - baseline", K1[r], BASE)
print()
for r in RANKS: show(f"r{r} @4000 - baseline", K4[r], BASE)
print("\nrank, at both checkpoints (Overall only):")
for lab, D in (("@1000", K1), ("@4000", K4)):
    for a, b in ((8, 4), (16, 8), (16, 4)):
        show(f"r{a} - r{b} {lab}", D[a], D[b], mets=[("Overall", ov)])

steps effect on test — step 1000 minus step 4000:
  r4  1000 - 4000                PF       n= 488  delta  -0.025  sigma_d 1.200  SE 0.054  p 0.651  90%CI [-0.114, +0.065]
  r4  1000 - 4000                SC       n= 488  delta  +0.523  sigma_d 1.921  SE 0.087  p 3.68e-09  90%CI [+0.379, +0.666]
  r4  1000 - 4000                Overall  n= 488  delta  +0.333  sigma_d 1.482  SE 0.067  p 9.82e-07  90%CI [+0.222, +0.443]
  r8  1000 - 4000                PF       n= 488  delta  +0.008  sigma_d 1.161  SE 0.053  p 0.876  90%CI [-0.078, +0.095]
  r8  1000 - 4000                SC       n= 488  delta  +0.414  sigma_d 1.921  SE 0.087  p 2.54e-06  90%CI [+0.271, +0.557]
  r8  1000 - 4000                Overall  n= 488  delta  +0.249  sigma_d 1.614  SE 0.073  p 0.000722  90%CI [+0.128, +0.369]
  r16  1000 - 4000               PF       n= 488  delta  +0.020  sigma_d 1.204  SE 0.054  p 0.707  90%CI [-0.069, +0.110]
  r16  1000 - 4000               SC       n= 488  delta  +0.514  sigma_d 1.903  SE 0

## 4. Rank contrasts
The three pairwise contrasts within a checkpoint carry two degrees of freedom, not three, and the two checkpoints score the same runs on the same rows, so inverse-variance pooling across all six would reuse the same information and understate the SE. Each contrast is reported with its own interval and the widest is quoted per checkpoint.

In [4]:
# The three pairwise contrasts within a checkpoint are algebraically dependent:
# (r8-r4) + (r16-r8) = (r16-r4) exactly, so they carry two degrees of freedom, not three.
# The two checkpoints are also the same runs scored on the same rows. Inverse-variance
# pooling across all six would count the same information repeatedly and understate the SE.
# The rank relationship is not monotone (at step 4000 r4->r8 is positive and r8->r16
# negative), so no single endpoint contrast bounds the others. Each pairwise contrast is
# reported with its own interval and the widest is quoted per checkpoint.
import itertools
for D, lab in ((K1, "@1000 (selected)"), (K4, "@4000")):
    widest = 0.0
    for a, b in itertools.combinations((4, 8, 16), 2):
        s = paired(D[b], D[a], I, ov)
        bound = abs(s["mean"]) + 1.96 * s["se"]
        widest = max(widest, bound)
        print(f"  r{a}->r{b:<3} {lab:<18} delta {s['mean']:+.4f}  SE {s['se']:.4f}  "
              f"95%CI [{s['mean'] - 1.96*s['se']:+.3f}, {s['mean'] + 1.96*s['se']:+.3f}]")
    print(f"  {lab}: no rank contrast is significant; widest pairwise bound {widest:.3f} Overall\n")
print("no detectable LoRA rank effect among ranks 4, 8 and 16 at either evaluated checkpoint")
print("trainable parameters: r4 2,553,600 | r8 5,107,200 | r16 10,214,400")
print("  38 attention blocks x 4 projections = 152 adapted modules; sum(d_in + d_out) per block = 16,800")

  r4->r8   @1000 (selected)   delta -0.0051  SE 0.0435  95%CI [-0.090, +0.080]
  r4->r16  @1000 (selected)   delta +0.0037  SE 0.0395  95%CI [-0.074, +0.081]
  r8->r16  @1000 (selected)   delta +0.0088  SE 0.0509  95%CI [-0.091, +0.109]
  @1000 (selected): no rank contrast is significant; widest pairwise bound 0.109 Overall

  r4->r8   @4000              delta +0.0789  SE 0.0603  95%CI [-0.039, +0.197]
  r4->r16  @4000              delta +0.0105  SE 0.0687  95%CI [-0.124, +0.145]
  r8->r16  @4000              delta -0.0684  SE 0.0537  95%CI [-0.174, +0.037]
  @4000: no rank contrast is significant; widest pairwise bound 0.197 Overall

no detectable LoRA rank effect among ranks 4, 8 and 16 at either evaluated checkpoint
trainable parameters: r4 2,553,600 | r8 5,107,200 | r16 10,214,400
  38 attention blocks x 4 projections = 152 adapted modules; sum(d_in + d_out) per block = 16,800


## 5. TOST against the pre-registered ±0.15 bound
Equivalence is declared only if both one-sided tests reject, i.e. the 90% CI lies entirely inside the bound. A CI straddling the bound is reported as inconclusive, not as equivalent.

In [5]:
BOUND = 0.15
def tost_clustered(A, B, idx, bound=BOUND):
    """Target-clustered TOST. Robustness companion to tost(); iid remains pre-specified."""
    d = np.array([ov(A[k]) - ov(B[k]) for k in idx]); m = d.mean(); n = len(d)
    g = defaultdict(list)
    for k, x in zip(idx, d): g[k.split("|")[0]].append(x)
    G = len(g)
    se = math.sqrt(sum((sum(v) - len(v) * m) ** 2 for v in g.values()) / n**2 * (G / (G - 1)))
    tc = stats.t.ppf(0.95, G - 1)
    p_lo = 1 - stats.t.cdf((m + bound) / se, G - 1)
    p_hi = stats.t.cdf((m - bound) / se, G - 1)
    return dict(mean=m, se=se, lo=m - tc * se, hi=m + tc * se, G=G), p_lo, p_hi, (p_lo < 0.05 and p_hi < 0.05)

def tost(A, B, idx, bound=BOUND):
    s = paired(A, B, idx, ov); n = s["n"]
    p_lo = 1 - stats.t.cdf((s["mean"] + bound) / s["se"], n - 1)
    p_hi = stats.t.cdf((s["mean"] - bound) / s["se"], n - 1)
    return s, p_lo, p_hi, (p_lo < 0.05 and p_hi < 0.05)

for lab, A in [(f"r{r} @1000", K1[r]) for r in RANKS] + [(f"r{r} @4000", K4[r]) for r in RANKS]:
    s, p_lo, p_hi, eq = tost(A, BASE, I)
    print(f"  {lab:12s} vs baseline  delta {s['mean']:+.3f}  90%CI [{s['lo']:+.3f}, {s['hi']:+.3f}]  "
          f"p(>-{BOUND})={p_lo:.3f}  p(<+{BOUND})={p_hi:.2g}  -> {'equivalent' if eq else 'not equivalent'}")
s, p_lo, p_hi, eq = tost(ZAD, ZBASE, IZ)
print(f"  {'r8 @1000, 0-ref':12s} vs base     delta {s['mean']:+.3f}  90%CI [{s['lo']:+.3f}, {s['hi']:+.3f}]  "
      f"p(>-{BOUND})={p_lo:.3f}  p(<+{BOUND})={p_hi:.2g}  -> {'equivalent' if eq else 'not equivalent'}")

  r4 @1000     vs baseline  delta -0.115  90%CI [-0.203, -0.027]  p(>-0.15)=0.254  p(<+0.15)=5e-07  -> not equivalent
  r8 @1000     vs baseline  delta -0.120  90%CI [-0.209, -0.031]  p(>-0.15)=0.287  p(<+0.15)=4e-07  -> not equivalent
  r16 @1000    vs baseline  delta -0.111  90%CI [-0.202, -0.020]  p(>-0.15)=0.239  p(<+0.15)=1.4e-06  -> not equivalent
  r4 @4000     vs baseline  delta -0.447  90%CI [-0.582, -0.313]  p(>-0.15)=1.000  p(<+0.15)=5.7e-13  -> not equivalent
  r8 @4000     vs baseline  delta -0.368  90%CI [-0.498, -0.239]  p(>-0.15)=0.997  p(<+0.15)=6e-11  -> not equivalent
  r16 @4000    vs baseline  delta -0.437  90%CI [-0.566, -0.308]  p(>-0.15)=1.000  p(<+0.15)=1.7e-13  -> not equivalent
  r8 @1000, 0-ref vs base     delta +0.064  90%CI [-0.032, +0.159]  p(>-0.15)=0.000  p(<+0.15)=0.069  -> not equivalent


## 6. Cluster-robust standard errors
Rows sharing an output target are not independent. The delta ICC is near zero for the prompting arms but not for the adapter, so every adapter contrast is re-tested with target-clustered errors.

In [6]:
def delta_icc(A, B, idx):
    g = defaultdict(list)
    for k in idx: g[k.split("|")[0]].append(ov(A[k]) - ov(B[k]))
    allv = np.concatenate([np.array(v) for v in g.values()]); kbar = np.mean([len(v) for v in g.values()])
    grand = allv.mean()
    msb = sum(len(v) * (np.mean(v) - grand)**2 for v in g.values()) / (len(g) - 1)
    msw = sum(((np.array(v) - np.mean(v))**2).sum() for v in g.values()) / (len(allv) - len(g))
    return (msb - msw) / (msb + (kbar - 1) * msw)

def cluster_se(A, B, idx):
    d = np.array([ov(A[k]) - ov(B[k]) for k in idx]); m = d.mean(); n = len(d)
    g = defaultdict(list)
    for k, x in zip(idx, d): g[k.split("|")[0]].append(x)
    G = len(g)
    num = sum((np.sum(np.array(v) - m))**2 for v in g.values())
    return m, d.std(ddof=1) / math.sqrt(n), math.sqrt(num / n**2 * (G / (G - 1))), G

print(f"  {'contrast':22s} {'ICC(delta)':>10s} {'SE iid':>8s} {'SE clust':>9s} {'t iid':>7s} {'t clust':>8s} {'p clust':>8s}")
for lab, A, B in ([(f"r{r} @1000 - base", K1[r], BASE) for r in RANKS]
                  + [(f"r{r} @4000 - base", K4[r], BASE) for r in RANKS]):
    icc = delta_icc(A, B, I); m, se_i, se_c, G = cluster_se(A, B, I)
    p = 2 * (1 - stats.norm.cdf(abs(m / se_c)))
    print(f"  {lab:22s} {icc:>10.3f} {se_i:>8.4f} {se_c:>9.4f} {m/se_i:>7.2f} {m/se_c:>8.2f} {p:>8.3f}")

  contrast               ICC(delta)   SE iid  SE clust   t iid  t clust  p clust
  r4 @1000 - base             0.060   0.0534    0.0548   -2.15    -2.09    0.036
  r8 @1000 - base             0.024   0.0539    0.0568   -2.22    -2.11    0.035
  r16 @1000 - base            0.091   0.0550    0.0587   -2.02    -1.89    0.059
  r4 @4000 - base             0.062   0.0818    0.0887   -5.47    -5.04    0.000
  r8 @4000 - base             0.105   0.0787    0.0912   -4.68    -4.04    0.000
  r16 @4000 - base            0.082   0.0784    0.0900   -5.57    -4.86    0.000


## 7. Stratification
Only on variables fixed before generation. `n_refs` and `ref_kind` come from the score rows; person/object provenance is derived from the instruction text. Never on baseline score.

In [7]:
PERSON = {"man","woman","men","women","person","people","boy","girl","boys","girls","guy","lady","businessman",
          "businesswoman","musician","musicians","child","children","kid","kids","baby","model","player","athlete",
          "worker","couple","gentleman","barber","tourist","tourists","doctor","doctors","assistant","assistants"}
_PREP = (r"\b(in|with|on|at|by|of|from|and|to|dressed|wearing|holding|using|reviewing|sitting|standing|"
         r"typing|sketching|working|reading|writing|playing|looking|smiling|drinking|reaching|leaning|posing|walking|browsing)\b")
def person_obj(instr):
    kinds = set()
    for p in re.split(r"(?<=\.)\s+", instr):
        if "<img>" not in p: continue
        m = re.search(r"(?:the\s+)?(.+?)\s+(?:is|are)\s+", p.lower())
        head = re.split(_PREP, m.group(1) if m else p.lower())[0].strip().split()
        kinds.add("person" if (head and head[-1] in PERSON) else "object")
    return "person" if kinds == {"person"} else ("object" if kinds == {"object"} else "mixed")

PROV = {k: person_obj(META[k]["instruction"]) for k in I}

def strata(A, B, key, order, idx=None):
    idx = idx or I
    print(f"  {key:12s} {'n':>4s} {'base SC':>8s} {'dPF':>8s} {'dSC':>8s} {'dOverall':>9s} {'p':>8s}")
    for lev in order:
        u = [k for k in idx if (PROV[k] if key == "provenance" else B[k][key]) == lev]
        if not u: continue
        d = np.array([ov(A[k]) - ov(B[k]) for k in u])
        print(f"  {str(lev):12s} {len(u):>4d} {np.mean([B[k]['SC'] for k in u]):>8.3f} "
              f"{np.mean([A[k]['PF'] - B[k]['PF'] for k in u]):>+8.3f} "
              f"{np.mean([A[k]['SC'] - B[k]['SC'] for k in u]):>+8.3f} "
              f"{d.mean():>+9.3f} {stats.ttest_1samp(d, 0)[1]:>8.3g}")

for lab, A in (("r8 @1000", K1[8]), ("r8 @4000", K4[8])):
    for key, order in (("n_refs", (1, 2, 3)), ("ref_kind", ("no_up", "mixed", "all_up")),
                       ("provenance", ("person", "mixed", "object"))):
        print(f"\n{lab} minus baseline, by {key}:"); strata(A, BASE, key, order)

print("\nref_kind and provenance are near-collinear, so neither cut identifies the mechanism on its own:")
tab = np.array([[sum(1 for k in I if BASE[k]["ref_kind"] == rk and PROV[k] == p)
                 for p in ("person", "mixed", "object")] for rk in ("no_up", "mixed", "all_up")])
print(f"  contingency (rows no_up/mixed/all_up, cols person/mixed/object):\n{tab}")
print(f"  Cramer's V = {math.sqrt(stats.chi2_contingency(tab)[0] / (len(I) * 2)):.3f}")


r8 @1000 minus baseline, by n_refs:
  n_refs          n  base SC      dPF      dSC  dOverall        p
  1             243    8.058   +0.037   -0.350    -0.205   0.0237
  2             194    8.155   +0.088   -0.165    -0.047    0.506
  3              51    7.980   +0.059   -0.020    +0.010    0.913

r8 @1000 minus baseline, by ref_kind:
  ref_kind        n  base SC      dPF      dSC  dOverall        p
  no_up         157    8.631   +0.178   -0.115    +0.046    0.441
  mixed         133    8.436   +0.023   +0.053    +0.043     0.51
  all_up        198    7.424   -0.010   -0.540    -0.360  0.00192

r8 @1000 minus baseline, by provenance:
  provenance      n  base SC      dPF      dSC  dOverall        p
  person        130    8.931   +0.192   -0.108    +0.055    0.407
  mixed         134    8.396   +0.022   +0.060    +0.047    0.476
  object        224    7.415   +0.004   -0.500    -0.321    0.002

r8 @4000 minus baseline, by n_refs:
  n_refs          n  base SC      dPF      dSC  dOvera

## 8. Validation curves and the two stopping signals
Diffusion validation loss is transcribed from the training logs (`lora_train_r*`, `VAL_EVERY=250`). GPT-4.1 PF/SC is scored on the same 98 val rows.

In [8]:
VLOSS = {  # transcribed from the training console logs
 "r4":  {250:0.34918, 500:0.34891, 750:0.34836, 1000:0.34784, 1250:0.34736, 1500:0.34696, 1750:0.34662,
         2000:0.34631, 3000:0.34543, 4000:0.34482},
 "r8":  {250:0.34919, 500:0.34895, 750:0.34842, 1000:0.34786, 1250:0.34736, 1500:0.34690, 1750:0.34653,
         2000:0.34619, 3000:0.34520, 4000:0.34450},
 "r16": {250:0.34919, 500:0.34896, 750:0.34842, 1000:0.34781, 1250:0.34724, 1500:0.34674, 1750:0.34631,
         2000:0.34592, 3000:0.34477, 4000:0.34395}}

print(f"{'rank':5s} {'step':>5s} {'val loss':>9s} {'GPT Overall':>12s} {'GPT PF':>7s} {'GPT SC':>7s} {'vs step 0':>10s} {'p':>7s}")
for r in ("r4", "r8", "r16"):
    b = np.mean([ov(VBASE[k]) for k in IV])
    print(f"{r:5s} {'0':>5s} {'-':>9s} {b:>12.3f} {np.mean([VBASE[k]['PF'] for k in IV]):>7.3f} "
          f"{np.mean([VBASE[k]['SC'] for k in IV]):>7.3f} {'-':>10s} {'-':>7s}")
    for s in (1000, 2000, 3000, 4000):
        D = VAL[r][s]; d = np.array([ov(D[k]) - ov(VBASE[k]) for k in IV])
        print(f"{'':5s} {s:>5d} {VLOSS[r][s]:>9.5f} {np.mean([ov(D[k]) for k in IV]):>12.3f} "
              f"{np.mean([D[k]['PF'] for k in IV]):>7.3f} {np.mean([D[k]['SC'] for k in IV]):>7.3f} "
              f"{d.mean():>+10.3f} {stats.ttest_1samp(d, 0)[1]:>7.3f}")
    g = [np.mean([ov(VAL[r][s][k]) for k in IV]) for s in (1000, 2000, 3000, 4000)]
    l = [VLOSS[r][s] for s in (1000, 2000, 3000, 4000)]
    print(f"{'':5s} argmin val loss {max(VLOSS[r], key=lambda s: -VLOSS[r][s])}, "
          f"argmax GPT Overall {(1000,2000,3000,4000)[int(np.argmax(g))]}, "
          f"Spearman(loss, Overall) {stats.spearmanr(l, g)[0]:+.2f}\n")

print("r8 with the early checkpoints, and the regional test that the val slice can support:")
PTS = {0: VBASE, **{s: VEARLY[s] for s in (250, 500, 750)}, **{s: VAL["r8"][s] for s in (1000, 2000, 3000, 4000)}}
U8 = sorted(set.intersection(*[set(d) for d in list(PTS.values()) + [VEARLY[1000]]]))
for s in sorted(PTS):
    print(f"  step {s:>4d}  Overall {np.mean([ov(PTS[s][k]) for k in U8]):.4f}")
# one definition of the early region, used by both contrasts. Step 750 is the lowest early
# point, so an inconsistent set moves the first p-value across 0.05 on its own.
EARLY_SET = (250, 500, 750, 1000)
early = np.array([np.mean([ov(PTS[s][k]) for s in EARLY_SET]) - ov(PTS[0][k]) for k in U8])
late  = np.array([np.mean([ov(PTS[s][k]) for s in (2000, 3000, 4000)])
                  - np.mean([ov(PTS[s][k]) for s in EARLY_SET]) for k in U8])
print(f"  early region {EARLY_SET} minus step 0 : {early.mean():+.3f}  p {stats.ttest_1samp(early,0)[1]:.4f}")
print(f"  late (>=2000) minus early (<=1000)          : {late.mean():+.3f}  p {stats.ttest_1samp(late,0)[1]:.4f}")
# the curve splices two runs at 750->1000; step 1000 exists in both, so the seam is measurable.
# the two runs agree at every observed point: val loss to <=1.0e-05 at every matched step and the
# step-1000 endpoints near-identically; the intermediate optimisation path was not observed. The
# step-1000 adapters correlate at 1.0000 (mean |delta| 9.85e-07 against rms 1.88e-04), so the
# seam measures generation and judge noise rather than divergence.
_sd = [ov(VEARLY[1000][k]) - ov(VAL["r8"][1000][k]) for k in U8]
_s = paired(VEARLY[1000], VAL["r8"][1000], U8, ov)
_clo, _chi = stats.chi2.interval(0.95, _s["n"] - 1)
print(f"  seam at step 1000, early run minus main run: {_s['mean']:+.3f}  "
      f"p {stats.ttest_1samp(_sd, 0)[1]:.3f}  sigma_d {_s['sd']:.3f} "
      f"95%CI [{_s['sd']*math.sqrt((_s['n']-1)/_chi):.3f}, {_s['sd']*math.sqrt((_s['n']-1)/_clo):.3f}] "
            f"against 0.966 for an identical config -> consistent with the noise floor, splice holds")
sd = np.mean([paired(PTS[a], PTS[b], U8, ov)["sd"] for a, b in ((250,500),(500,750),(750,1000),(1000,2000))])
z = (stats.norm.ppf(0.975) + stats.norm.ppf(0.80))**2
print(f"  checkpoint-to-checkpoint sigma_d {sd:.3f} -> difference giving ~80% power at n={len(U8)}: "
      f"{math.sqrt(z*sd**2/len(U8)):.3f}")

rank   step  val loss  GPT Overall  GPT PF  GPT SC  vs step 0       p
r4        0         -        7.739   7.898   7.816          -       -
       1000   0.34784        7.857   7.969   7.898     +0.118   0.278
       2000   0.34631        7.722   7.908   7.684     -0.016   0.908
       3000   0.34543        7.654   8.000   7.469     -0.085   0.657
       4000   0.34482        7.665   7.990   7.520     -0.074   0.687
      argmin val loss 4000, argmax GPT Overall 1000, Spearman(loss, Overall) +0.80

r8        0         -        7.739   7.898   7.816          -       -
       1000   0.34786        7.943   7.980   8.031     +0.204   0.077
       2000   0.34619        7.744   8.153   7.602     +0.005   0.977
       3000   0.34520        7.600   7.959   7.500     -0.139   0.445
       4000   0.34450        7.574   8.041   7.327     -0.165   0.365
      argmin val loss 4000, argmax GPT Overall 1000, Spearman(loss, Overall) +1.00

r16       0         -        7.739   7.898   7.816          - 

## 9. Val-to-test transfer
Whether the val slice predicted the ordering of checkpoints, and whether it predicted the level.

In [9]:
print(f"  {'contrast':26s} {'val (n=98)':>12s} {'test (n=488)':>14s} {'same sign':>10s}")
for r in RANKS:
    rk = f"r{r}"
    for lab, va, vb, ta, tb in (("@1000 - baseline", VAL[rk][1000], VBASE, K1[r], BASE),
                                ("@1000 - @4000",    VAL[rk][1000], VAL[rk][4000], K1[r], K4[r])):
        dv = np.mean([ov(va[k]) - ov(vb[k]) for k in IV])
        dt = np.mean([ov(ta[k]) - ov(tb[k]) for k in I])
        print(f"  r{r:<3d} {lab:21s} {dv:>+12.3f} {dt:>+14.3f} {str(np.sign(dv)==np.sign(dt)):>10s}")

# why the level did not transfer. No positive delta clustering was detected on this validation
# slice: the val delta ICC estimate is negative, so there is no evidence that clustering inflates
# the val standard error. On 98 rows across 20 targets this does not exclude cluster dependence;
# the test-split adapter contrasts carry the cluster-robust sensitivity checks.
ntg = len({k.split("|")[0] for k in IV})
icc = delta_icc(VAL["r8"][1000], VBASE, IV)
sv = paired(VAL["r8"][1000], VBASE, IV, ov); st = paired(K1[8], BASE, I, ov)
gap = sv["mean"] - st["mean"]; gse = math.sqrt(sv["se"]**2 + st["se"]**2)
print(f"\n  val slice: {len(IV)} rows over {ntg} targets ({len(IV)/ntg:.1f} rows/target), delta ICC {icc:+.3f}")
print(f"  val SE {sv['se']:.3f} vs test SE {st['se']:.3f}")
print(f"  r8 @1000 - baseline: val {sv['mean']:+.3f}, test {st['mean']:+.3f}, gap {gap:+.3f} "
      f"= {gap/gse:.1f} SE (p {2*(1-stats.norm.cdf(abs(gap/gse))):.3f})")
print(f"  base Overall on the two slices: val {np.mean([ov(VBASE[k]) for k in IV]):.3f}, "
      f"test {np.mean([ov(BASE[k]) for k in I]):.3f}")
print("  step 1000 was selected as the val argmax, so its val estimate carries selection bias;")
print("  the ordering of checkpoints transferred, the level against the baseline did not.")

  contrast                     val (n=98)   test (n=488)  same sign
  r4   @1000 - baseline            +0.118         -0.115      False
  r4   @1000 - @4000               +0.192         +0.333       True
  r8   @1000 - baseline            +0.204         -0.120      False
  r8   @1000 - @4000               +0.368         +0.249       True
  r16  @1000 - baseline            +0.111         -0.111      False
  r16  @1000 - @4000               +0.343         +0.326       True

  val slice: 98 rows over 20 targets (4.9 rows/target), delta ICC -0.124
  val SE 0.114 vs test SE 0.054
  r8 @1000 - baseline: val +0.204, test -0.120, gap +0.324 = 2.6 SE (p 0.010)
  base Overall on the two slices: val 7.739, test 7.963
  step 1000 was selected as the val argmax, so its val estimate carries selection bias;
  the ordering of checkpoints transferred, the level against the baseline did not.


## 10. Zero-reference adaptation tax
Adapter against base with references removed from both: a zero-reference adaptation-tax check, testing whether the adapter changes performance once reference conditioning is gone. The localisation claim is tested directly by the paired difference-of-differences computed in this section over the 488 common rows — (adapter − base) with references minus (adapter − base) without — which is −0.184 Overall, SE 0.075, p 0.015, 95% CI [−0.331, −0.036]; the degradation is therefore concentrated in the reference-conditioned setting rather than merely significant in one condition and not the other. The paired adapter-minus-base delta is the result; absolute SC and Overall are not conventional text-to-image quality, since the judge still scores against the withheld reference subjects.

In [10]:
print("adapter minus base, no references supplied to either:")
show("r8 @1000 - base, 0-ref", ZAD, ZBASE, idx=IZ)

print("\n2x2, Overall:")
print(f"  {'':22s} {'base':>8s} {'adapter':>9s} {'delta':>9s} {'p':>9s}")
for lab, B_, A_, idx in (("with references", BASE, K1[8], I), ("no references", ZBASE, ZAD, IZ)):
    s = paired(A_, B_, idx, ov)
    print(f"  {lab:22s} {np.mean([ov(B_[k]) for k in idx]):>8.3f} "
          f"{np.mean([ov(A_[k]) for k in idx]):>9.3f} {s['mean']:>+9.3f} {s['p']:>9.3g}")

print("\nthe tax by provenance, against the same cut of the with-reference SC damage:")
print(f"  {'stratum':10s} {'n':>4s} {'0-ref dOverall':>15s} {'p':>8s} {'with-ref dSC':>13s}")
for st in ("person", "mixed", "object"):
    u = [k for k in IZ if PROV.get(k) == st]
    if not u: continue
    d = np.array([ov(ZAD[k]) - ov(ZBASE[k]) for k in u])
    wr = np.mean([K1[8][k]["SC"] - BASE[k]["SC"] for k in u if k in I])
    print(f"  {st:10s} {len(u):>4d} {d.mean():>+15.3f} {stats.ttest_1samp(d,0)[1]:>8.3g} {wr:>+13.3f}")

adapter minus base, no references supplied to either:
  r8 @1000 - base, 0-ref         PF       n= 488  delta  +0.025  sigma_d 1.070  SE 0.048  p 0.612  90%CI [-0.055, +0.104]
  r8 @1000 - base, 0-ref         SC       n= 488  delta  +0.090  sigma_d 1.603  SE 0.073  p 0.215  90%CI [-0.029, +0.210]
  r8 @1000 - base, 0-ref         Overall  n= 488  delta  +0.064  sigma_d 1.279  SE 0.058  p 0.271  90%CI [-0.032, +0.159]

2x2, Overall:
                             base   adapter     delta         p
  with references           7.963     7.843    -0.120    0.0269
  no references             5.727     5.791    +0.064     0.271

the tax by provenance, against the same cut of the with-reference SC damage:
  stratum       n  0-ref dOverall        p  with-ref dSC
  person      130          +0.065    0.499        -0.108
  mixed       134          +0.170   0.0887        +0.060
  object      224          -0.000    0.996        -0.500


In [11]:
# zero-reference interaction: (adapter - base | refs) - (adapter - base | no refs), uid-paired
_I = sorted(set(K1[8]) & set(BASE) & set(ZAD) & set(ZBASE))
_w = np.array([ov(K1[8][k]) - ov(BASE[k])  for k in _I])
_z = np.array([ov(ZAD[k])   - ov(ZBASE[k]) for k in _I])
_d = _w - _z; _n = len(_d); _m = _d.mean(); _se = _d.std(ddof=1) / math.sqrt(_n)
_g = defaultdict(list)
for k, x in zip(_I, _d): _g[k.split("|")[0]].append(x)
_G = len(_g)
_secl = math.sqrt(sum((sum(v) - len(v) * _m) ** 2 for v in _g.values()) / _n**2 * (_G / (_G - 1)))
_p  = stats.ttest_rel(_w, _z)[1]
_pc = 2 * (1 - stats.t.cdf(abs(_m / _secl), _G - 1))
_tc = stats.t.ppf(0.975, _n - 1)
print(f"zero-reference interaction  n {_n}  mean {_m:+.4f}")
print(f"  iid       SE {_se:.4f}  p {_p:.4f}  95% CI [{_m-_tc*_se:+.4f}, {_m+_tc*_se:+.4f}]")
print(f"  clustered SE {_secl:.4f}  p {_pc:.4f} over {_G} target scenes")
print("  the adapter effect is worse with reference conditioning present")

# step-1000 rank-vs-rank TOSTs, the values the notes report
print(f"\nstep-1000 rank equivalence (bound +/-{BOUND}):")
for _a, _b in ((4, 8), (4, 16), (8, 16)):
    _idx = sorted(set(K1[_a]) & set(K1[_b]))
    _s, _lo, _hi, _eq = tost(K1[_a], K1[_b], _idx)
    _c, _clo, _chi, _ceq = tost_clustered(K1[_a], K1[_b], _idx)
    print(f"  r{_a}-r{_b}: {_s['mean']:+.4f}  iid 90% CI [{_s['lo']:+.4f}, {_s['hi']:+.4f}]  "
          f"TOST p {max(_lo,_hi):.4f}  equivalent {_eq}")
    print(f"           clustered 90% CI [{_c['lo']:+.4f}, {_c['hi']:+.4f}]  "
          f"TOST p {max(_clo,_chi):.4f}  equivalent {_ceq}  ({_c['G']} targets)")

zero-reference interaction  n 488  mean -0.1836
  iid       SE 0.0752  p 0.0150  95% CI [-0.3313, -0.0358]
  clustered SE 0.0803  p 0.0244 over 100 target scenes
  the adapter effect is worse with reference conditioning present

step-1000 rank equivalence (bound +/-0.15):
  r4-r8: +0.0051  iid 90% CI [-0.0666, +0.0767]  TOST p 0.0005  equivalent True
           clustered 90% CI [-0.0705, +0.0806]  TOST p 0.0010  equivalent True  (100 targets)
  r4-r16: -0.0037  iid 90% CI [-0.0689, +0.0614]  TOST p 0.0001  equivalent True
           clustered 90% CI [-0.0744, +0.0670]  TOST p 0.0004  equivalent True  (100 targets)
  r8-r16: -0.0088  iid 90% CI [-0.0927, +0.0751]  TOST p 0.0029  equivalent True
           clustered 90% CI [-0.0908, +0.0732]  TOST p 0.0026  equivalent True  (100 targets)


## 11. Failure-mode composition
Whether fine-tuning introduces new failure modes or produces more of the existing ones. Rationale text is matched with keyword patterns; counts are indicative, the comparison across runs is the point.

In [12]:
MODES = {"identity": r"\bidentit|different (person|man|woman)",
         "missing":  r"not (present|visible|shown)|missing|absent",
         "colour":   r"\bcolou?r",
         "shape":    r"\bshape|\bform\b|proportion|design"}
print(f"  {'run':22s} {'low-SC rows':>12s} " + " ".join(f"{k:>10s}" for k in MODES))
ROWS = [("baseline", BASE)] + [(f"r{r} @{s}", (K1 if s == 1000 else K4)[r])
                               for r in RANKS for s in (1000, 4000)]
for lab, D in ROWS:
    low = [D[k] for k in I if D[k]["SC"] <= 5]
    pct = [100 * sum(1 for x in low if re.search(v, str(x["SC_reason"]), re.I)) / max(len(low), 1) for v in MODES.values()]
    print(f"  {lab:22s} {len(low):>7d}/{len(I):<4d} " + " ".join(f"{v:>9.0f}%" for v in pct))

for lab, D in [(f"r{r} @{s}", (K1 if s == 1000 else K4)[r]) for r in RANKS for s in (1000, 4000)]:
    w = sum(1 for k in I if D[k]["SC"] < BASE[k]["SC"]); b = sum(1 for k in I if D[k]["SC"] > BASE[k]["SC"])
    print(f"\n  {lab} vs baseline: SC worse {w}, better {b}, unchanged {len(I)-w-b}; "
          f"catastrophic (<=-4) {sum(1 for k in I if D[k]['SC']-BASE[k]['SC'] <= -4)}; "
          f"SC>=8 falling below 6: {sum(1 for k in I if BASE[k]['SC']>=8 and D[k]['SC']<6)}")

  run                     low-SC rows   identity    missing     colour      shape
  baseline                    49/488         88%        18%        57%        65%
  r4 @1000                    68/488         88%        16%        57%        69%
  r4 @4000                    97/488         88%        18%        62%        71%
  r8 @1000                    62/488         87%        24%        56%        79%
  r8 @4000                    91/488         86%        18%        66%        67%
  r16 @1000                   62/488         81%        15%        48%        69%
  r16 @4000                   94/488         79%        22%        66%        69%

  r4 @1000 vs baseline: SC worse 136, better 93, unchanged 259; catastrophic (<=-4) 20; SC>=8 falling below 6: 21

  r4 @4000 vs baseline: SC worse 193, better 65, unchanged 230; catastrophic (<=-4) 52; SC>=8 falling below 6: 48

  r8 @1000 vs baseline: SC worse 135, better 94, unchanged 259; catastrophic (<=-4) 20; SC>=8 falling below 6: 20